In [1]:
import pandas as pd
import numpy as np

In [17]:
df_224 = pd.read_csv("../features/auto_cov_features.csv")
df_225 = pd.read_csv("../features/composition_features.csv")
df_226 = pd.read_csv("../features/one_lead_bigram_asa.csv")
df_221_223 = pd.read_csv("../features/pssm_features.csv")
g_data_df = pd.read_csv("../data/g_data.csv", header=None)

In [18]:
g_data_df.head()

,0,1,2,3
0,1,Fold1,>P23453,MSGEVLSQNEIDALLSAISTGEMDAEELKKEEKEKKVKVYDFKRAL...
1,1,Fold1,>P07373,MTTKKTSPDLLLVIITLLLLTIGLIMVYSASAVWADYKFDDSFFFA...
2,1,Fold1,>P12921,MIIWINGAFGSGKTQTAFELHRRLNPSYVYDPQKMGFALRSMVPQE...
3,1,Fold1,>P19579,MRRKLTFQEKLLIFIKKTKKKNPRYVAIVLPLIAVILIAATWVQRT...
4,1,Fold1,>A2RMA8,MQNLNKTEKTFFGQPRGLLTLFQTEFWERFSYYGMRAILVYYLYAL...


In [20]:
features_456 = df_224.merge(df_225, on='Unnamed: 0').merge(df_226, on='Unnamed: 0')

In [21]:
features_456 = features_456.rename(columns={"Unnamed: 0": "Protein_ID"})
features_456.shape

(1911, 36)

In [23]:
# Extract ID and fold, clean up '>' from IDs
g_data_fold = g_data_df[[2, 1]].copy()
g_data_fold[2] = g_data_fold[2].astype(str).str.strip().str.replace('>', '', regex=False)
g_data_fold.columns = [features_456.columns[0], 'target']


In [31]:
g_data_fold.head()

,Protein_ID,target
0,P23453,Fold1
1,P07373,Fold1
2,P12921,Fold1
3,P19579,Fold1
4,A2RMA8,Fold1


In [ ]:
df_221_223[df_221_223.columns[0]] = df_221_223[df_221_223.columns[0]].astype(str).str.strip()

# Merge
merged_df = pd.merge(df_221_223, g_data_fold, on=df_221_223.columns[0], how='left')

# Reorder to place 'target' after Protein_ID
cols = list(merged_df.columns)
reordered_cols = [cols[0], 'target'] + [c for c in cols[1:] if c != 'target']
merged_df = merged_df[reordered_cols]

In [38]:
# Ensure pssm IDs are strings and strip any extra whitespace
features_456[features_456.columns[0]] = features_456[features_456.columns[0]].astype(str).str.strip()


In [45]:
merged_df = pd.merge(merged_df, features_456, on=features_456.columns[0], how='left')

In [46]:
merged_df.head()

,Protein_ID,target,F1,F2,F3,F4,F5,F6,F7,F8,...,AC_26,AC_27,AC_28,AC_29,AC_30,Composition_phi,Composition_psi,Composition_theta,Composition_tau,OLB_ASA
0,Q55055,Fold3,-109.0,-156.0,-257.0,-212.0,-419.0,-171.0,-125.0,-332.0,...,0.044063,0.182650,0.141760,0.039419,0.187204,0.037283,0.154646,0.000180,0.013856,0.000197
1,Q04959,Fold1,48.0,-346.0,-320.0,-409.0,-524.0,-52.0,-296.0,-61.0,...,0.171523,0.007637,0.350981,0.164488,0.007394,0.046045,0.294395,0.000118,0.014920,0.000277
2,Q08429,Fold1,-75.0,-107.0,-101.0,-139.0,-256.0,-55.0,-55.0,-199.0,...,0.206525,0.084684,0.126607,0.198947,0.082603,0.036383,0.351760,0.000114,0.012426,0.000212
3,Q83GS8,Fold3,-698.0,-1137.0,-905.0,-1052.0,-2128.0,-974.0,-726.0,-1659.0,...,0.128312,0.056503,0.190841,0.122777,0.056882,0.045349,0.291179,0.000359,0.014921,0.000254
4,Q8G864,Fold3,-307.0,-413.0,-649.0,-569.0,-1103.0,-396.0,-312.0,-807.0,...,0.264833,0.023684,0.108447,0.261128,0.023220,0.041002,0.462962,0.000289,0.010621,0.000154


In [48]:
merged_df.to_csv("../features/merged_df.csv")